# Fault Detection — Fine-tuning Prithvi-EO 2.0 (600M)

**Objective 1A**: Fine-tune Prithvi-EO 2.0 to identify active faults from Sentinel-2 imagery.

Pipeline:
- Backbone: `prithvi_eo_v2_600m` (pretrained on 4.2M HLS samples)
- Decoder: `UperNetDecoder` with multi-scale features from layers [7, 15, 23, 31]
- Loss: Focal (handles ~1-5% fault pixel imbalance)
- Input: `[B, T=1, C=6, H=128, W=128]` patches
- Metrics: mIoU, F1, pixel accuracy

**Before running**: Upload `patches.zip` to `Google Drive/fault_detection_data/patches.zip`

In [ ]:
# ── 1. INSTALL DEPENDENCIES ──────────────────────────────────────────────────
# terratorch: official TerraTorch framework for Prithvi fine-tuning
# albumentations: data augmentation (used by TerraTorch transforms)
# einops: tensor rearranging (used internally by Prithvi)

!pip install terratorch albumentations einops timm -q

# Verify GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── 2. MOUNT GOOGLE DRIVE ────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── 3. COPY DATA FROM DRIVE → /content/ ──────────────────────────────────────
# /content/ is local SSD — much faster I/O than Drive during training

import os

DRIVE_ZIP = '/content/drive/MyDrive/fault_detection_data/patches.zip'
LOCAL_ZIP = '/content/patches.zip'
LOCAL_DIR = '/content/data'

if not os.path.exists(f'{LOCAL_DIR}/patches/splits/train.txt'):
    print('Copying from Drive...')
    !cp "{DRIVE_ZIP}" "{LOCAL_ZIP}"
    print('Extracting...')
    !unzip -q "{LOCAL_ZIP}" -d "{LOCAL_DIR}"
    print('Done.')
else:
    print('Data already extracted.')

# Verify
with open(f'{LOCAL_DIR}/patches/splits/train.txt') as f:
    train_names = [l.strip() for l in f if l.strip()]
with open(f'{LOCAL_DIR}/patches/splits/val.txt') as f:
    val_names = [l.strip() for l in f if l.strip()]
with open(f'{LOCAL_DIR}/patches/splits/test.txt') as f:
    test_names = [l.strip() for l in f if l.strip()]

print(f'Train: {len(train_names):,}  Val: {len(val_names):,}  Test: {len(test_names):,}')

# Region breakdown
for region in ['carrizo', 'mojave', 'bay_area']:
    n = sum(1 for n in train_names if n.startswith(region))
    print(f'  {region}: {n} train patches')

In [ ]:
# ── 4. CONFIG ────────────────────────────────────────────────────────────────

# Paths
PATCHES_DIR    = f'{LOCAL_DIR}/patches'
IMAGES_DIR     = f'{PATCHES_DIR}/images'
LABELS_DIR     = f'{PATCHES_DIR}/labels'
SPLITS_DIR     = f'{PATCHES_DIR}/splits'
CHECKPOINT_DIR = '/content/checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Model
BACKBONE    = 'prithvi_eo_v2_600_tl'  # 600M — _tl = transfer learning (pretrained weights)
NUM_CLASSES = 2                         # 0=no fault, 1=fault
NUM_FRAMES  = 1                         # single Sentinel-2 composite (no time series)
IMG_SIZE    = 128

# Training — matches landslide4sense config from Prithvi-EO-2.0 repo
BATCH_SIZE   = 32    # A100 40GB handles this easily at 128x128
LR           = 5e-5  # AdamW lr from landslide config
WEIGHT_DECAY = 0.1
EPOCHS       = 80
PATIENCE     = 10    # early stopping patience (checks every 2 epochs)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print(f'Backbone: {BACKBONE}')

In [ ]:
# ── 5. DATASET + DATA MODULE ─────────────────────────────────────────────────
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import lightning as L


class FaultPatchDataset(Dataset):
    """
    Loads 128x128 .npy patch pairs.
    Returns a dict matching TerraTorch's expected batch format:
      image : [C=6, T=1, H=128, W=128]  float32  — Prithvi expects [C, T, H, W]
      mask  : [H=128, W=128]             int64
    """
    def __init__(self, split_file, images_dir, labels_dir, augment=False):
        with open(split_file) as f:
            self.names = [l.strip() for l in f if l.strip()]
        self.images_dir = Path(images_dir)
        self.labels_dir = Path(labels_dir)
        self.augment    = augment

    def __len__(self):
        return len(self.names)

    def __getitem__(self, idx):
        name = self.names[idx]
        img = np.load(self.images_dir / f'{name}.npy').astype(np.float32)  # [6, 128, 128]
        lbl = np.load(self.labels_dir / f'{name}.npy').astype(np.int64)    # [128, 128]

        if self.augment:
            if np.random.random() > 0.5:
                img = img[:, :, ::-1].copy()
                lbl = lbl[:, ::-1].copy()
            if np.random.random() > 0.5:
                img = img[:, ::-1, :].copy()
                lbl = lbl[::-1, :].copy()

        # [6, 128, 128] -> [6, 1, 128, 128]  (C, T, H, W) — required by Prithvi
        img = img[:, np.newaxis, :, :]

        return {
            "image": torch.from_numpy(img),
            "mask":  torch.from_numpy(lbl),
        }


class FaultDataModule(L.LightningDataModule):
    def __init__(self, patches_dir, batch_size, num_workers=4):
        super().__init__()
        self.patches_dir = patches_dir
        self.batch_size  = batch_size
        self.num_workers = num_workers

    def setup(self, stage=None):
        # Named train_dataset / val_dataset / test_dataset — required by TerraTorch
        self.train_dataset = FaultPatchDataset(
            f'{self.patches_dir}/splits/train.txt',
            f'{self.patches_dir}/images',
            f'{self.patches_dir}/labels',
            augment=True,
        )
        self.val_dataset = FaultPatchDataset(
            f'{self.patches_dir}/splits/val.txt',
            f'{self.patches_dir}/images',
            f'{self.patches_dir}/labels',
        )
        self.test_dataset = FaultPatchDataset(
            f'{self.patches_dir}/splits/test.txt',
            f'{self.patches_dir}/images',
            f'{self.patches_dir}/labels',
        )

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size,
                          shuffle=True, num_workers=self.num_workers, pin_memory=True)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size,
                          shuffle=False, num_workers=self.num_workers, pin_memory=True)

    def test_dataloader(self):
        return DataLoader(self.test_dataset, batch_size=self.batch_size,
                          shuffle=False, num_workers=self.num_workers, pin_memory=True)


datamodule = FaultDataModule(PATCHES_DIR, batch_size=BATCH_SIZE)
datamodule.setup()
print(f'Train: {len(datamodule.train_dataset):,}  Val: {len(datamodule.val_dataset):,}  Test: {len(datamodule.test_dataset):,}')

sample = datamodule.train_dataset[0]
img, lbl = sample["image"], sample["mask"]
print(f'Image shape: {tuple(img.shape)}  →  expect (6, 1, 128, 128)')
print(f'Label shape: {tuple(lbl.shape)}  →  expect (128, 128)')
print(f'Fault pixels: {lbl.float().mean()*100:.2f}%')

In [ ]:
# ── 6. VISUALIZE SAMPLE PATCHES ──────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 6, figsize=(18, 6))
axes[0, 0].set_ylabel('Image (RGB)', fontsize=10)
axes[1, 0].set_ylabel('Fault mask', fontsize=10)

for i in range(6):
    sample = datamodule.train_dataset[i * 20]
    img, lbl = sample["image"], sample["mask"]
    # img is [C=6, T=1, H, W] — take T=0, bands R/G/B = indices 2,1,0
    rgb = img[[2, 1, 0], 0].permute(1, 2, 0).numpy()
    rgb = np.clip(rgb, 0, 1)

    axes[0, i].imshow(rgb)
    axes[0, i].set_title(datamodule.train_dataset.names[i * 20].split('_')[0], fontsize=8)
    axes[0, i].axis('off')

    axes[1, i].imshow(lbl.numpy(), cmap='Reds', vmin=0, vmax=1)
    axes[1, i].set_title(f'fault%: {lbl.float().mean()*100:.1f}%', fontsize=8)
    axes[1, i].axis('off')

plt.suptitle('Sample training patches (top=Sentinel-2 RGB, bottom=fault label)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── 7. BUILD MODEL ───────────────────────────────────────────────────────────
import types
import terratorch
import terratorch.models
import terratorch.models.backbones
from terratorch.tasks import SemanticSegmentationTask

model_args = {
    'backbone'          : BACKBONE,
    'decoder'           : 'UperNetDecoder',
    'num_classes'       : NUM_CLASSES,
    'rescale'           : True,
    'head_dropout'      : 0.1,
    'decoder_channels'  : 256,
    'head_channel_list' : [128, 64],
    'necks': [
        {'name': 'SelectIndices', 'indices': [7, 15, 23, 31]},
        {'name': 'ReshapeTokensToImage'},
    ],
    'backbone_kwargs': {
        'num_frames' : NUM_FRAMES,
        'pretrained' : True,
        'bands'      : ['BLUE', 'GREEN', 'RED', 'NIR_BROAD', 'SWIR_1', 'SWIR_2'],
    },
}

task = SemanticSegmentationTask(
    model_args     = model_args,
    model_factory  = 'EncoderDecoderFactory',
    loss           = 'ce',
    ignore_index   = -1,
    freeze_backbone= False,
    freeze_decoder = False,
)

# ── Override training/validation steps to bypass TerraTorch's broken CE path ──
_ce = torch.nn.CrossEntropyLoss()

def _logits(self, x):
    out = self.model(x)
    return out.output if hasattr(out, 'output') else out

def training_step(self, batch, batch_idx):
    logits = _logits(self, batch["image"])
    loss   = _ce(logits, batch["mask"].long())
    self.log("train/loss", loss, on_step=True, on_epoch=True, prog_bar=True)
    return loss

def validation_step(self, batch, batch_idx, dataloader_idx=0):
    logits = _logits(self, batch["image"])
    loss   = _ce(logits, batch["mask"].long())
    self.log("val/loss", loss, on_step=False, on_epoch=True, prog_bar=True, sync_dist=True)
    return loss

task.training_step  = types.MethodType(training_step,  task)
task.validation_step = types.MethodType(validation_step, task)
print("Steps overridden with plain CrossEntropyLoss")

total_params     = sum(p.numel() for p in task.parameters())
trainable_params = sum(p.numel() for p in task.parameters() if p.requires_grad)
print(f'Total params:     {total_params:,}')
print(f'Trainable params: {trainable_params:,}')

In [ ]:
# ── 8. OPTIMIZER + SCHEDULER ─────────────────────────────────────────────────
#
# TerraTorch's SemanticSegmentationTask inherits configure_optimizers from
# its parent. We override it here to set AdamW + CosineAnnealingLR,
# matching the landslide4sense config exactly.
#
import torch.optim as optim

def configure_optimizers(self):
    optimizer = optim.AdamW(
        self.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=EPOCHS,
        eta_min=1e-7,
    )
    return {
        'optimizer': optimizer,
        'lr_scheduler': {'scheduler': scheduler, 'interval': 'epoch'},
    }

# Bind to task instance
import types
task.configure_optimizers = types.MethodType(configure_optimizers, task)
print('Optimizer configured: AdamW lr=5e-5, CosineAnnealingLR')

In [ ]:
# ── 9. TRAIN ─────────────────────────────────────────────────────────────────
from lightning.pytorch.callbacks import (
    ModelCheckpoint,
    LearningRateMonitor,
    EarlyStopping,
    RichProgressBar,
)

checkpoint_cb = ModelCheckpoint(
    dirpath    = CHECKPOINT_DIR,
    filename   = 'prithvi600m-fault-{epoch:02d}-{val/loss:.4f}',
    monitor    = 'val/loss',
    mode       = 'min',
    save_top_k = 1,
    save_last  = True,
)

early_stop_cb = EarlyStopping(
    monitor       = 'val/loss',
    patience      = PATIENCE,
    mode          = 'min',
    check_finite  = False,   # don't stop on NaN — let training recover
)

trainer = L.Trainer(
    max_epochs              = EPOCHS,
    accelerator             = 'gpu',
    devices                 = 1,
    precision               = 'bf16-mixed',
    gradient_clip_val       = 1.0,
    check_val_every_n_epoch = 2,
    log_every_n_steps       = 10,
    enable_checkpointing    = True,
    default_root_dir        = CHECKPOINT_DIR,
    callbacks=[
        checkpoint_cb,
        early_stop_cb,
        LearningRateMonitor(logging_interval='epoch'),
        RichProgressBar(),
    ],
)

trainer.fit(task, datamodule=datamodule)
print(f'\nBest checkpoint: {checkpoint_cb.best_model_path}')
print(f'Best val/loss:   {checkpoint_cb.best_model_score:.4f}')

In [ ]:
# ── 10. EVALUATE ON TEST SET ─────────────────────────────────────────────────
# Load best checkpoint and run test split
# TerraTorch's SemanticSegmentationTask logs:
#   test/loss, test/Multiclass_Jaccard_Index (mIoU), test/Multiclass_F1Score

best_ckpt = checkpoint_cb.best_model_path
print(f'Loading checkpoint: {best_ckpt}')

test_results = trainer.test(task, datamodule=datamodule, ckpt_path=best_ckpt)
print('\n=== TEST RESULTS ===')
for k, v in test_results[0].items():
    print(f'  {k}: {v:.4f}')

In [ ]:
# ── 11. MANUAL METRICS (mIoU, F1, pixel accuracy) ────────────────────────────
# Compute per-patch metrics manually to report exactly what the proposal requires:
# mIoU, F1, pixel accuracy  (Objective 1A evaluation)

task.eval().to(DEVICE)

tp_total = fp_total = fn_total = tn_total = 0

with torch.no_grad():
    for batch in datamodule.test_dataloader():
        imgs = batch["image"].to(DEVICE)  # [B, 1, 6, 128, 128]
        lbls = batch["mask"].to(DEVICE)   # [B, 128, 128]

        out    = task(imgs)
        logits = out.output if hasattr(out, 'output') else out
        preds  = logits.argmax(dim=1)  # [B, 128, 128]

        tp_total += ((preds == 1) & (lbls == 1)).sum().item()
        fp_total += ((preds == 1) & (lbls == 0)).sum().item()
        fn_total += ((preds == 0) & (lbls == 1)).sum().item()
        tn_total += ((preds == 0) & (lbls == 0)).sum().item()

iou      = tp_total / (tp_total + fp_total + fn_total + 1e-8)
f1       = 2 * tp_total / (2 * tp_total + fp_total + fn_total + 1e-8)
pix_acc  = (tp_total + tn_total) / (tp_total + fp_total + fn_total + tn_total + 1e-8)
precision = tp_total / (tp_total + fp_total + 1e-8)
recall    = tp_total / (tp_total + fn_total + 1e-8)

print('=== OBJECTIVE 1A — TEST SET METRICS ===')
print(f'  mIoU (fault class):  {iou:.4f}')
print(f'  F1 score:            {f1:.4f}')
print(f'  Pixel accuracy:      {pix_acc:.4f}')
print(f'  Precision:           {precision:.4f}')
print(f'  Recall:              {recall:.4f}')

In [ ]:
# ── 12. VISUALIZE PREDICTIONS ────────────────────────────────────────────────
# Show image / ground truth / prediction for 6 test patches

task.eval().to(DEVICE)

batch = next(iter(datamodule.test_dataloader()))
imgs_batch = batch["image"]
lbls_batch = batch["mask"]

with torch.no_grad():
    out    = task(imgs_batch.to(DEVICE))
    logits = out.output if hasattr(out, 'output') else out
    preds  = logits.argmax(dim=1).cpu()  # [B, 128, 128]

n_show = 6
fig, axes = plt.subplots(3, n_show, figsize=(18, 9))
axes[0, 0].set_ylabel('Sentinel-2 RGB', fontsize=10)
axes[1, 0].set_ylabel('Ground Truth',   fontsize=10)
axes[2, 0].set_ylabel('Prediction',     fontsize=10)

for i in range(n_show):
    rgb = imgs_batch[i, 0, [2, 1, 0]].permute(1, 2, 0).numpy()
    rgb = np.clip(rgb, 0, 1)

    axes[0, i].imshow(rgb)
    axes[0, i].axis('off')

    axes[1, i].imshow(lbls_batch[i].numpy(), cmap='Reds', vmin=0, vmax=1)
    axes[1, i].set_title(f'GT fault%: {lbls_batch[i].float().mean()*100:.1f}%', fontsize=7)
    axes[1, i].axis('off')

    axes[2, i].imshow(preds[i].numpy(), cmap='Reds', vmin=0, vmax=1)
    axes[2, i].set_title(f'Pred fault%: {preds[i].float().mean()*100:.1f}%', fontsize=7)
    axes[2, i].axis('off')

plt.suptitle('Test Predictions  |  Top: Image  |  Mid: Ground Truth  |  Bot: Model', fontsize=12)
plt.tight_layout()
plt.savefig(f'{CHECKPOINT_DIR}/predictions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 13. SAVE CHECKPOINT + RESULTS TO DRIVE ───────────────────────────────────
import shutil

DRIVE_OUTPUT = '/content/drive/MyDrive/fault_detection_data/checkpoints'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

# Copy best checkpoint
shutil.copy(checkpoint_cb.best_model_path, DRIVE_OUTPUT)
shutil.copy(f'{CHECKPOINT_DIR}/predictions.png', DRIVE_OUTPUT)

# Save metrics
import json
metrics = {
    'mIoU'           : round(iou, 4),
    'F1'             : round(f1, 4),
    'pixel_accuracy' : round(pix_acc, 4),
    'precision'      : round(precision, 4),
    'recall'         : round(recall, 4),
    'best_val_loss'  : float(checkpoint_cb.best_model_score),
    'backbone'       : BACKBONE,
    'epochs_trained' : trainer.current_epoch,
}
with open(f'{DRIVE_OUTPUT}/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print('Saved to Drive:')
for item in os.listdir(DRIVE_OUTPUT):
    print(f'  {DRIVE_OUTPUT}/{item}')